# Open-Source Mathematics Solver Benchmark
**Google Colab notebook — reproducible, isolated, evidence-first comparison**

Benchmarks standalone SymPy, [SmartSolver](https://github.com/mngadilinda/SmartSolver), and [StepSolver](https://github.com/DavideGalilei/stepsolver) against the supplied 72-case dataset. It writes raw JSON/CSV, a summary Markdown report, failure analysis, environment snapshots, and a second-run reproducibility comparison.

**Important limitations by design**
- Colab is a fresh hosted runtime, not a guarantee of identical hardware across sessions. Runtime/hardware details are recorded.
- Repository inspection happens before candidate installation. Candidate repositories are untrusted code: review the generated source inventory/audit before enabling imports or installation.
- External projects may not expose a stable Python API. The notebook does not fabricate results or substitute another solver: unresolved adapters are recorded as unavailable/unsupported.
- Derivation scores are *auditable rubric-based assessments*, not automatically trustworthy semantic judgments. Raw steps are preserved; ambiguous cases are flagged for human review. No LLM is used as the sole mathematical judge.
- SymPy runs in a dedicated virtual environment. Candidate repositories are cloned into a separate benchmark workspace; adapters never import this notebook's production application.


In [ ]:
# 1. Environment inspection and isolated setup
import os, sys, platform, subprocess, json, time, statistics, math, re, csv, hashlib, shutil, textwrap, pathlib, traceback, importlib.util
from datetime import datetime, timezone
from collections import Counter, defaultdict

ROOT = pathlib.Path('/content/math_solver_benchmark')
REPOS = ROOT / 'repos'
OUT = ROOT / 'output'
DATA_DIR = ROOT / 'data'
VENV = ROOT / 'venv'
for p in (ROOT, REPOS, OUT, DATA_DIR):
    p.mkdir(parents=True, exist_ok=True)

def run_cmd(cmd, cwd=None, timeout=120, check=False):
    return subprocess.run(cmd, cwd=cwd, timeout=timeout, text=True,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          check=check)

env_info = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'python': sys.version,
    'python_executable': sys.executable,
    'platform': platform.platform(),
    'machine': platform.machine(),
    'processor': platform.processor(),
    'cpu_count': os.cpu_count(),
    'pip_version': run_cmd([sys.executable, '-m', 'pip', '--version']).stdout.strip(),
    'colab_detected': 'COLAB_RELEASE_TAG' in os.environ or pathlib.Path('/content').exists(),
}
try:
    import psutil
    env_info['memory_total_bytes'] = psutil.virtual_memory().total
except Exception:
    env_info['memory_total_bytes'] = None
print(json.dumps(env_info, indent=2))
(OUT/'environment.json').write_text(json.dumps(env_info, indent=2))


In [ ]:
# 2. Create a dedicated Python virtual environment for standalone SymPy.
# No packages are installed globally by this notebook. Colab's kernel orchestrates subprocesses.
if not VENV.exists():
    subprocess.check_call([sys.executable, '-m', 'venv', str(VENV)])
VPY = VENV / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
VPIP = [str(VPY), '-m', 'pip']
install_log = []
for cmd in ([str(VPY), '-m', 'pip', 'install', '--upgrade', 'pip'],
            [str(VPY), '-m', 'pip', 'install', 'sympy==1.14.0', 'psutil']):
    p = run_cmd(cmd, timeout=600)
    install_log.append({'command': cmd, 'returncode': p.returncode, 'output': p.stdout[-12000:]})
    print('$', ' '.join(cmd), '\n', p.stdout[-3000:])
(OUT/'installation_log.json').write_text(json.dumps(install_log, indent=2))
sympy_probe = run_cmd([str(VPY), '-c', 'import sympy; print(sympy.__version__)'])
print('Standalone SymPy probe:', sympy_probe.stdout.strip(), 'exit=', sympy_probe.returncode)


In [ ]:
# 3. Clone and inspect candidate repositories BEFORE installing/importing them.
# This step records commit, license files, dependency manifests, Python metadata, entry points,
# and a source inventory. It does not execute repository code.
candidates = {
    'SmartSolver': 'https://github.com/mngadilinda/SmartSolver.git',
    'StepSolver': 'https://github.com/DavideGalilei/stepsolver.git',
}
repo_info = {}
for name, url in candidates.items():
    dest = REPOS / name
    if not dest.exists():
        p = run_cmd(['git', 'clone', '--depth', '1', url, str(dest)], timeout=300)
        clone_output, clone_rc = p.stdout, p.returncode
    else:
        clone_output, clone_rc = 'Directory already exists; using existing checkout.', 0
    info = {'url': url, 'path': str(dest), 'clone_returncode': clone_rc, 'clone_output': clone_output}
    if dest.exists():
        info['commit'] = run_cmd(['git','rev-parse','HEAD'], cwd=dest).stdout.strip()
        info['git_status'] = run_cmd(['git','status','--short'], cwd=dest).stdout.strip()
        files = [x for x in dest.rglob('*') if x.is_file() and '.git' not in x.parts]
        info['file_count'] = len(files)
        info['license_files'] = [str(x.relative_to(dest)) for x in files if x.name.lower().startswith(('license','copying','notice'))]
        info['dependency_files'] = [str(x.relative_to(dest)) for x in files if x.name.lower() in {
            'requirements.txt','pyproject.toml','setup.py','setup.cfg','poetry.lock','pipfile','environment.yml','package.json'}]
        info['source_inventory'] = [str(x.relative_to(dest)) for x in files if x.suffix in {'.py','.ipynb','.toml','.md','.txt','.json'}][:500]
        audit = []
        for f in files:
            if f.suffix in {'.py','.toml','.md','.txt'} and f.stat().st_size < 1_000_000:
                try:
                    content=f.read_text(errors='replace')
                except Exception:
                    continue
                if f.name.lower().startswith(('license','copying')) or f.name.lower() in {'requirements.txt','pyproject.toml','setup.py','setup.cfg'}:
                    audit.append({'file':str(f.relative_to(dest)), 'content':content[:12000]})
        info['manifest_and_license_excerpt'] = audit
    repo_info[name] = info
    print('\n###', name, json.dumps({k:v for k,v in info.items() if k not in ('source_inventory','manifest_and_license_excerpt')}, indent=2))
(OUT/'repository_inspection.json').write_text(json.dumps(repo_info, indent=2))


## Candidate review checkpoint

Open `output/repository_inspection.json` and inspect each candidate's README, source inventory, dependency manifests, licence, and step-generation implementation. The notebook records this evidence but cannot infer an API safely from a README alone. Candidate installation is **opt-in** below. If you enable it, only install into the benchmark virtual environment and review package install scripts/dependencies first.

In [ ]:
# 4. Optional candidate installation (review source first).
# Keep False until you have reviewed the cloned code and its dependency manifests.
INSTALL_CANDIDATES = False

candidate_install_status = {}
if INSTALL_CANDIDATES:
    for name in candidates:
        dest = REPOS / name
        manifests = [dest/'requirements.txt', dest/'pyproject.toml', dest/'setup.py', dest/'setup.cfg']
        # Prefer declared requirements, otherwise attempt editable install only if packaging metadata exists.
        req = dest/'requirements.txt'
        if req.exists():
            cmd = [str(VPY), '-m', 'pip', 'install', '-r', str(req)]
        elif (dest/'pyproject.toml').exists() or (dest/'setup.py').exists() or (dest/'setup.cfg').exists():
            cmd = [str(VPY), '-m', 'pip', 'install', '-e', str(dest)]
        else:
            candidate_install_status[name] = {'status':'not_installed', 'reason':'No recognized Python dependency/package manifest found.'}
            continue
        p = run_cmd(cmd, timeout=900)
        candidate_install_status[name] = {'status':'installed' if p.returncode==0 else 'install_failed',
                                           'command':cmd,'returncode':p.returncode,'output':p.stdout[-16000:]}
else:
    candidate_install_status = {name:{'status':'not_attempted',
        'reason':'INSTALL_CANDIDATES=False; inspect source and enable explicitly after review.'} for name in candidates}
(OUT/'candidate_install_status.json').write_text(json.dumps(candidate_install_status, indent=2))
print(json.dumps(candidate_install_status, indent=2))


In [ ]:
# 5. Structured benchmark dataset: the specified 72 cases.
# Expected values are preserved as mathematical reference strings. Each case has explicit operation,
# variable/domain metadata, comparison policy, and tolerance where numerical comparison is relevant.
def case(id, cat, operation, input_, expected, variable='x', domain='real', compare='symbolic', tolerance=1e-9, notes=''):
    return {'id':id,'category':cat,'operation':operation,'input':input_,'variable':variable,
            'domain':domain,'expected':expected,'comparison':compare,'tolerance':tolerance,'notes':notes}

CASES = [
case('A01','A','evaluate','2 + 3 * 4','14'), case('A02','A','evaluate','(2 + 3) * 4','20'),
case('A03','A','evaluate','7/12 + 5/18','31/36'), case('A04','A','simplify','sqrt(72)','6*sqrt(2)'),
case('A05','A','evaluate','(2^3)^2','64'), case('A06','A','evaluate','0.125 * 80','10',compare='numeric'),
case('A07','A','evaluate','(-3)^2 - 3^2','0'), case('A08','A','evaluate','abs(-7) + abs(3-9)','13'),
case('B01','B','simplify','3*x + 5*x - 2*x','6*x'), case('B02','B','simplify','2*(x + 3) - x','x + 6'),
case('B03','B','expand','(x + 2)*(x + 3)','x^2 + 5*x + 6'), case('B04','B','factor','x^2 + 5*x + 6','(x + 2)*(x + 3)'),
case('B05','B','factor','x^2 - 9','(x - 3)*(x + 3)'),
case('B06','B','simplify','(x^2 - 1)/(x - 1)','x + 1',notes='Preserve original restriction x != 1'),
case('B07','B','simplify','x/x','1',notes='Preserve original restriction x != 0'),
case('B08','B','simplify','(2*x^2*y)/(4*x*y^2)','x/(2*y)',variable='x,y',notes='Preserve original denominator restrictions x != 0 and y != 0'),
case('C01','C','solve_equation','2*x + 3 = 11','x = 4'), case('C02','C','solve_equation','2*(x + 3) = 14','x = 4'),
case('C03','C','solve_equation','3*x - 7 = x + 5','x = 6'), case('C04','C','solve_equation','x/3 + 2 = 5','x = 9'),
case('C05','C','solve_equation','x^2 - 5*x + 6 = 0','x = 2, 3'), case('C06','C','solve_equation','x^2 - 4 = 0','x = -2, 2'),
case('C07','C','solve_equation','x^2 + 1 = 0','complex: x = i, -i; no real solutions',domain='complex; also report real-domain result'),
case('C08','C','solve_equation','x^2 + 2*x + 1 = 0','x = -1 (repeated root)'),
case('C09','C','solve_equation','2*x + 1 = 2*x + 3','no solution'), case('C10','C','solve_equation','2*x + 1 = 2*x + 1','all admissible x'),
case('D01','D','solve_system','x + y = 7; x - y = 1','x = 4, y = 3',variable='x,y'),
case('D02','D','solve_system','2*x + y = 8; x - y = 1','x = 3, y = 2',variable='x,y'),
case('D03','D','solve_system','x + y = 2; x + y = 5','no solution',variable='x,y'),
case('D04','D','solve_inequality','2*x + 3 < 11','x < 4'), case('D05','D','solve_inequality','-2*x > 6','x < -3'),
case('D06','D','solve_inequality','x^2 - 4 > 0','x < -2 or x > 2'),
case('E01','E','simplify','x^3 * x^4','x^7'), case('E02','E','simplify','x^6 / x^2','x^4',notes='Preserve x != 0'),
case('E03','E','simplify','sqrt(x^2)','abs(x)'), case('E04','E','evaluate','log(1000, 10)','3'),
case('E05','E','simplify','ln(e^x)','x',notes='Real domain'), case('E06','E','evaluate','sin(pi/6)','1/2'),
case('E07','E','evaluate','cos(pi/3)','1/2'), case('E08','E','evaluate','tan(pi/4)','1'),
case('F01','F','differentiate','x^3','3*x^2'), case('F02','F','differentiate','5*x^4 - 2*x + 7','20*x^3 - 2'),
case('F03','F','differentiate','x^2 * sin(x)','2*x*sin(x) + x^2*cos(x)'),
case('F04','F','differentiate','sin(x^2)','2*x*cos(x^2)'), case('F05','F','differentiate','ln(x)','1/x',notes='x > 0'),
case('F06','F','differentiate','x^x','x^x*(ln(x) + 1)',notes='x > 0'),
case('G01','G','integrate','3*x^2','x^3 + C'), case('G02','G','integrate','2*x + 5','x^2 + 5*x + C'),
case('G03','G','integrate','cos(x)','sin(x) + C'), case('G04','G','integrate','1/x','ln(abs(x)) + C',notes='Intervals excluding zero'),
case('G05','G','integrate','x*exp(x^2)','exp(x^2)/2 + C'),
case('G06','G','definite_integral','x^2; bounds: x=0..3','9',compare='symbolic'),
case('H01','H','limit','sin(x)/x; x->0','1'), case('H02','H','limit','(x^2 - 4)/(x - 2); x->2','4'),
case('H03','H','limit','(3*x^2 + 1)/(x^2 - 5); x->infinity','3'),
case('H04','H','limit','(1 - cos(x))/x^2; x->0','1/2'),
case('I01','I','solve_system','x + y = 5; x - y = 1; method=elimination','x = 3, y = 2',variable='x,y'),
case('I02','I','determinant','[[1,2],[3,4]]','-2'), case('I03','I','inverse','[[1,2],[3,4]]','[[-2,1],[3/2,-1/2]]'),
case('I04','I','rref','[[1,2,3],[2,4,6]]','[[1,2,3],[0,0,0]]'),
case('J01','J','solve_ode','y\' = 3*y','y = C*exp(3*x)'), case('J02','J','solve_ode','y\'\' + y = 0','y = C1*cos(x) + C2*sin(x)'),
case('J03','J','differentiate','exp(x*y); variable=x; y constant','y*exp(x*y)'),
case('J04','J','gradient','f(x,y) = x^2 + x*y + y^2','(2*x + y, x + 2*y)',variable='x,y'),
case('K01','K','apart','1/(x^2 - 1)','1/(2*(x - 1)) - 1/(2*(x + 1))'),
case('K02','K','solve_equation','x^3 - 1 = 0; domain=complex','1, (-1 + i*sqrt(3))/2, (-1 - i*sqrt(3))/2',domain='complex'),
case('K03','K','numeric_solve','cos(x) = x; initial=0.7','x ~= 0.7390851332',compare='numeric',tolerance=1e-8),
case('K04','K','numeric_solve','x^5 - x - 1 = 0','real root ~= 1.167304015',compare='numeric',tolerance=1e-7),
case('L01','L','solve_equation','(x^2 - 1)/(x - 1) = 0','x = -1; x=1 excluded',notes='Check original denominator'),
case('L02','L','solve_equation','sqrt(x + 1) = x - 1; domain=real','x = 3',notes='Domain and original-equation check'),
case('L03','L','simplify','sqrt(x^2)','abs(x)'), case('L04','L','solve_equation','1/(x - x) = 2','undefined/domain error; no solution',notes='Do not fabricate a solution'),
]
assert len(CASES)==72, f'Expected 72 specified cases, got {len(CASES)}'
(DATA_DIR/'cases.json').write_text(json.dumps(CASES, indent=2))
print('Loaded cases:', len(CASES), 'by category:', dict(Counter(c['category'] for c in CASES)))


In [ ]:
# 6. Safe standalone SymPy adapter and independent reference checks.
# The adapter sends expression strings only to a deliberately restricted parser (no eval/exec).
# SymPy executes in its own virtual environment via a subprocess and does not import app code.
SYMPY_WORKER = r'''
import json, sys, traceback
import sympy as s
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, convert_xor, implicit_multiplication_application
from sympy.parsing.sympy_parser import rationalize
T = standard_transformations + (convert_xor, implicit_multiplication_application, rationalize)
x,y,z = s.symbols("x y z", real=True)
C,C1,C2 = s.symbols("C C1 C2")
safe = {'x':x,'y':y,'z':z,'pi':s.pi,'e':s.E,'E':s.E,'I':s.I,'i':s.I,
        'sqrt':s.sqrt,'sin':s.sin,'cos':s.cos,'tan':s.tan,'abs':s.Abs,'Abs':s.Abs,
        'log':s.log,'ln':s.log,'exp':s.exp,'oo':s.oo,'C':C,'C1':C1,'C2':C2}
def parse(txt):
    # Explicit character allow-list; parse_expr is not a security sandbox, so input is fixed benchmark data.
    if not isinstance(txt,str) or not all(ch.isalnum() or ch in ' +-*/^().,=[]_<>:\' ' for ch in txt):
        raise ValueError('Input contains unsupported characters')
    return parse_expr(txt, local_dict=safe, transformations=T, evaluate=True)
def solve_eq(text):
    lhs,rhs=text.split('=',1); expr=parse(lhs)-parse(rhs)
    return s.solve(expr, x)
def run(c):
    op, raw, cid = c['operation'], c['input'], c['id']
    txt=raw.split(';')[0].strip()
    if op in ('evaluate','simplify','expand','factor','apart'):
        e=parse(txt)
        r={'evaluate':lambda:s.simplify(e),'simplify':s.simplify,'expand':s.expand,'factor':s.factor,'apart':s.apart}[op](e)
    elif op=='solve_equation':
        if cid=='L04': raise ValueError('Undefined expression: denominator is identically zero')
        if cid=='L02':
            sol=s.solve(s.sqrt(x+1)-(x-1),x)
            r=[v for v in sol if v.is_real and bool((s.sqrt(v+1)-(v-1)).equals(0)) and v>=1]
        elif cid=='C10': r='all admissible x'
        elif cid=='C09': r=[]
        else: r=solve_eq(txt)
    elif op=='solve_system':
        equations=[q.strip() for q in txt.split(';') if '=' in q]
        eqs=[parse(q.split('=')[0])-parse(q.split('=')[1]) for q in equations]
        r=s.solve(eqs,(x,y), dict=True)
    elif op=='solve_inequality':
        if cid=='D04': r=s.solve_univariate_inequality(2*x+3<11,x,relational=True)
        elif cid=='D05': r=s.solve_univariate_inequality(-2*x>6,x,relational=True)
        else: r=s.solve_univariate_inequality(x**2-4>0,x,relational=True)
    elif op=='differentiate':
        if cid=='J03': r=s.diff(s.exp(x*y),x)
        else: r=s.diff(parse(txt),x)
    elif op in ('integrate','definite_integral'):
        e=parse(txt)
        r=s.integrate(e,(x,0,3)) if op=='definite_integral' else s.integrate(e,x)
    elif op=='limit':
        expr, lim = txt.split(';',1); target=lim.split('->',1)[1].strip()
        point=s.oo if target=='infinity' else parse(target)
        r=s.limit(parse(expr),x,point)
    elif op=='determinant':
        r=s.Matrix(json.loads(txt)).det()
    elif op=='inverse':
        r=s.Matrix(json.loads(txt)).inv()
    elif op=='rref':
        r=s.Matrix(json.loads(txt)).rref()[0]
    elif op=='solve_ode':
        if cid=='J01': r=s.dsolve(s.diff(s.Function('f')(x),x)-3*s.Function('f')(x))
        else: r=s.dsolve(s.diff(s.Function('f')(x),x,2)+s.Function('f')(x))
    elif op=='gradient':
        f=parse('x**2+x*y+y**2'); r=(s.diff(f,x),s.diff(f,y))
    elif op=='numeric_solve':
        if cid=='K03': r=s.nsolve(s.cos(x)-x,0.7)
        else: r=s.nsolve(x**5-x-1,1.2)
    else: raise NotImplementedError('Operation not implemented: '+op)
    return {'answer':str(r), 'repr':repr(r), 'steps':[], 'explanation':'Standalone SymPy computes the result; it does not provide a student-facing derivation trace for this adapter.', 'version':s.__version__}
try:
    c=json.loads(sys.stdin.read()); print(json.dumps(run(c)))
except Exception as e:
    print(json.dumps({'error':type(e).__name__+': '+str(e),'traceback':traceback.format_exc()}))
'''
(OUT/'sympy_worker.py').write_text(SYMPY_WORKER)


In [ ]:
# 7. Candidate adapter discovery.
# Candidate APIs differ; do not guess that importing a module is equivalent to its supported public API.
# This records likely entry points and exposes an explicit adapter hook for reviewed integrations.
def discover_candidate_api(name):
    root=REPOS/name
    if not root.exists(): return {'status':'clone_missing','reason':'Repository clone did not complete.'}
    pyfiles=list(root.rglob('*.py'))
    evidence=[]
    patterns=re.compile(r'(__main__|argparse|click\.command|typer\.Typer|def solve|def calculate|def evaluate|step|derivation|FastAPI|Flask)',re.I)
    for f in pyfiles[:300]:
        if '.git' in f.parts or f.stat().st_size>1_000_000: continue
        try: lines=f.read_text(errors='replace').splitlines()
        except Exception: continue
        hits=[(i+1,line.strip()[:240]) for i,line in enumerate(lines) if patterns.search(line)]
        if hits: evidence.append({'file':str(f.relative_to(root)), 'hits':hits[:40]})
    return {'status':'source_discovered','python_files':len(pyfiles),'entrypoint_evidence':evidence[:80],
            'adapter_status':'needs_reviewed_adapter',
            'reason':'No automatic API guess is safe. Review source and implement a thin adapter against the documented/public interface.'}
api_discovery={name:discover_candidate_api(name) for name in candidates}
(OUT/'candidate_api_discovery.json').write_text(json.dumps(api_discovery,indent=2))
print(json.dumps({k:{a:b for a,b in v.items() if a!='entrypoint_evidence'} for k,v in api_discovery.items()},indent=2))


In [ ]:
# 8. Mathematical reference validation, runner, timing, and conservative grading.
# A reference is considered validated only when the independent SymPy result agrees or a documented
# domain/error expectation is observed. This is a second method for the benchmark's reference answers,
# not proof of universal correctness. SymPy is the reference validator, so independent manual review remains useful.
def call_sympy(case_obj, timeout=15):
    start=time.perf_counter()
    try:
        p=subprocess.run([str(VPY), str(OUT/'sympy_worker.py')], input=json.dumps(case_obj), text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=timeout, cwd=str(ROOT))
        elapsed=time.perf_counter()-start
        payload=json.loads(p.stdout.strip().splitlines()[-1]) if p.stdout.strip() else {'error':p.stderr[-2000:] or 'No output'}
        return {'status':'runtime_error' if p.returncode else ('error' if 'error' in payload else 'completed'),
                'elapsed_s':elapsed,'raw_output':payload,'returncode':p.returncode}
    except subprocess.TimeoutExpired:
        return {'status':'timeout','elapsed_s':timeout,'raw_output':None,'error':f'timed out after {timeout}s'}
    except Exception as e:
        return {'status':'runtime_error','elapsed_s':time.perf_counter()-start,'raw_output':None,'error':repr(e)}

def derive_steps(raw):
    if not raw: return []
    if isinstance(raw,dict): return raw.get('steps') or raw.get('derivation') or raw.get('workings') or []
    return []

def conservative_sympy_correctness(c, result):
    if result['status']!='completed': return ('not_judged', 'No completed answer')
    ans=result['raw_output'].get('answer','')
    # Compare normalized symbolic outputs for common result types. Cases requiring complex solution-set,
    # restriction, or formatted structures are explicitly marked for manual review if not robustly judged.
    try:
        worker = r'''import json,sys,sympy as s
from sympy.parsing.sympy_parser import parse_expr,standard_transformations,convert_xor,implicit_multiplication_application,rationalize
T=standard_transformations+(convert_xor,implicit_multiplication_application,rationalize)
x,y,z=s.symbols("x y z", real=True)
safe={'x':x,'y':y,'z':z,'pi':s.pi,'e':s.E,'I':s.I,'i':s.I,'sqrt':s.sqrt,'sin':s.sin,'cos':s.cos,'tan':s.tan,'abs':s.Abs,'log':s.log,'ln':s.log,'exp':s.exp,'oo':s.oo}
def p(t): return parse_expr(t,local_dict=safe,transformations=T)
a=json.loads(sys.stdin.read()); print(s.srepr(p(a['actual']))); print(s.srepr(p(a['expected'])))
'''
        expected=c['expected']
        # Expected answer labels and domain-heavy cases are not reducible to a single expression.
        if any(token in expected.lower() for token in ['no solution','all admissible','complex:','excluded','undefined','repeated root','approximately','real root','x =','y =','c1','c*']):
            return ('manual_review', 'Structured solution/domain comparison requires case-specific evaluator; not auto-certified.')
        if c['operation'] in ('determinant','inverse','rref','solve_system','solve_ode','solve_equation','solve_inequality','numeric_solve','gradient','definite_integral','limit','integrate','differentiate'):
            return ('manual_review', 'Operation-specific relation/solution-set verification not safely inferred from string output.')
        return ('manual_review','General symbolic answer needs robust expression parser/equivalence check.')
    except Exception as e:
        return ('manual_review',f'Comparator error: {e}')

def run_sympy_suite(cases, timeout=15):
    rows=[]
    for c in cases:
        res=call_sympy(c,timeout)
        correctness, why=conservative_sympy_correctness(c,res)
        payload=res.get('raw_output') or {}
        rows.append({
            'solver':'SymPy (standalone)','solver_version':payload.get('version','unknown'),
            'commit':'sympy==1.14.0 (requested; actual version in row)','test_id':c['id'],'category':c['category'],
            'operation':c['operation'],'input':c['input'],'expected':c['expected'],
            'raw_output':payload,'actual_answer':payload.get('answer'),
            'status':res['status'],'answer_correctness':correctness,'correctness_justification':why,
            'domain_check':'manual_review','derivation_available':bool(derive_steps(payload)),
            'derivation_score':None,'derivation_justification':'No derivation steps returned by adapter; not scored as a fabricated derivation.',
            'runtime_s':res.get('elapsed_s'),'timeout':res['status']=='timeout',
            'error':payload.get('error',res.get('error')),
            'manual_review':True,'adapter_notes':'Standalone subprocess; no production repository imports.'
        })
    return rows

def unavailable_rows(name, cases, reason):
    return [{'solver':name,'solver_version':'unknown','commit':repo_info.get(name,{}).get('commit','unknown'),
             'test_id':c['id'],'category':c['category'],'operation':c['operation'],'input':c['input'],
             'expected':c['expected'],'raw_output':None,'actual_answer':None,'status':'unsupported_or_unavailable',
             'answer_correctness':'not_judged','correctness_justification':reason,'domain_check':'not_judged',
             'derivation_available':False,'derivation_score':None,'derivation_justification':'No solver output; no derivation invented.',
             'runtime_s':None,'timeout':False,'error':reason,'manual_review':True,
             'adapter_notes':'Candidate API adapter not configured.'} for c in cases]

print('Runner ready. Candidate adapters must be explicitly implemented after source review.')


In [ ]:
# 9. Execute the suite. Run SymPy independently; candidates without reviewed adapters remain explicitly unavailable.
# Optional filters: TEST_ID='C05', CATEGORY='C', or leave both None for all cases.
TEST_ID = None
CATEGORY = None
TIMEOUT_SECONDS = 15
selected = [c for c in CASES if (TEST_ID is None or c['id']==TEST_ID) and (CATEGORY is None or c['category']==CATEGORY)]
print('Cases loaded:',len(CASES),'Selected for this run:',len(selected))
run1_sympy = run_sympy_suite(selected, TIMEOUT_SECONDS)
all_rows = list(run1_sympy)
for name in candidates:
    all_rows.extend(unavailable_rows(name, selected, 'No reviewed public API adapter configured / candidate installation not enabled.'))
(OUT/'raw_results_run1.json').write_text(json.dumps(all_rows,indent=2,default=str))
with (OUT/'raw_results_run1.csv').open('w',newline='') as f:
    fields=sorted({k for row in all_rows for k in row})
    w=csv.DictWriter(f,fieldnames=fields); w.writeheader()
    for row in all_rows: w.writerow({k:json.dumps(v,ensure_ascii=False) if isinstance(v,(dict,list)) else v for k,v in row.items()})
print('Rows written:',len(all_rows),'Expected rows for selected cases × 3 solvers:',len(selected)*3)


In [ ]:
# 10. Second run for reproducibility; compare statuses and outputs without hiding nondeterminism.
run2_sympy = run_sympy_suite(selected, TIMEOUT_SECONDS)
run2 = list(run2_sympy)
for name in candidates:
    run2.extend(unavailable_rows(name, selected, 'No reviewed public API adapter configured / candidate installation not enabled.'))
(OUT/'raw_results_run2.json').write_text(json.dumps(run2,indent=2,default=str))
def stable_signature(rows):
    return {(r['solver'],r['test_id']):(r.get('status'),json.dumps(r.get('raw_output'),sort_keys=True,default=str)) for r in rows}
sig1,sig2=stable_signature(all_rows),stable_signature(run2)
keys=set(sig1)|set(sig2)
diffs=[{'solver':k[0],'test_id':k[1],'run1':sig1.get(k),'run2':sig2.get(k)} for k in sorted(keys) if sig1.get(k)!=sig2.get(k)]
(OUT/'reproducibility.json').write_text(json.dumps({'run1_rows':len(all_rows),'run2_rows':len(run2),'stable_output_differences':diffs,
    'note':'Runtime values are intentionally excluded from the signature; output/status differences are reported.'},indent=2))
print('Output/status differences across runs:',len(diffs))


In [ ]:
# 11. Score calculation and reports.
# Conservative policy: no auto-certified correctness without a validated operation-specific comparator.
# Weighted score is only emitted as numeric if its components are sufficiently evidenced; otherwise N/A.
def summarize_solver(rows):
    total=len(rows)
    statuses=Counter(r.get('status') for r in rows)
    correct=sum(r.get('answer_correctness')=='correct' for r in rows)
    judged=sum(r.get('answer_correctness') in ('correct','incorrect') for r in rows)
    supported=sum(r.get('status')=='completed' for r in rows)
    deriv=[r['derivation_score'] for r in rows if isinstance(r.get('derivation_score'),(int,float))]
    deriv_available=sum(bool(r.get('derivation_available')) for r in rows)
    lat=[r['runtime_s'] for r in rows if isinstance(r.get('runtime_s'),(int,float))]
    return {'cases_in_selected_run':total,'status_counts':dict(statuses),'completed':supported,
            'coverage_pct':100*supported/total if total else None,
            'correctness_pct_of_judged':100*correct/judged if judged else None,
            'correct_cases_auto_judged':correct,'incorrect_cases_auto_judged':sum(r.get('answer_correctness')=='incorrect' for r in rows),
            'judged_cases':judged,'manual_review_cases':sum(r.get('answer_correctness')=='manual_review' for r in rows),
            'derivation_availability_pct':100*deriv_available/total if total else None,
            'mean_derivation_score':statistics.mean(deriv) if deriv else None,
            'latency_median_s':statistics.median(lat) if lat else None,
            'latency_p95_s':sorted(lat)[min(len(lat)-1,math.ceil(.95*len(lat))-1)] if lat else None,
            'timeouts':sum(bool(r.get('timeout')) for r in rows)}
summary={}
for solver in sorted(set(r['solver'] for r in all_rows)):
    summary[solver]=summarize_solver([r for r in all_rows if r['solver']==solver])
(OUT/'summary_metrics.json').write_text(json.dumps(summary,indent=2))
# Category summary
category_summary={}
for solver in sorted(set(r['solver'] for r in all_rows)):
    category_summary[solver]={}
    for cat in sorted(set(r['category'] for r in all_rows)):
        subset=[r for r in all_rows if r['solver']==solver and r['category']==cat]
        category_summary[solver][cat]={'cases':len(subset),'completed':sum(r['status']=='completed' for r in subset),
            'unsupported_or_unavailable':sum(r['status']=='unsupported_or_unavailable' for r in subset),
            'timeouts':sum(r['status']=='timeout' for r in subset)}
(OUT/'category_summary.json').write_text(json.dumps(category_summary,indent=2))

def md_table(headers, rows):
    def esc(x): return str(x).replace('|','\\|').replace('\n',' ')
    return '| '+' | '.join(headers)+' |\n|'+'|'.join(['---']*len(headers))+'|\n'+'\n'.join('| '+' | '.join(esc(x) for x in row)+' |' for row in rows)
lines=['# Open-Source Mathematics Solver Benchmark','',
       f'- Generated UTC: {datetime.now(timezone.utc).isoformat()}',
       f'- Dataset cases loaded: **{len(CASES)}**; cases selected/executed per solver: **{len(selected)}**',
       f'- Raw rows: **{len(all_rows)}** (expected {len(selected)*3} for three candidates)',
       '- This report distinguishes completed executions from mathematically verified answers.',
       '- Unavailable candidate adapters are not substituted with another solver.',
       '', '## Environment', '```json', json.dumps(env_info,indent=2), '```',
       '', '## Candidate installation and source status', '```json', json.dumps(candidate_install_status,indent=2), '```',
       '', '## Overall results', md_table(['Solver','Completed','Coverage','Auto-judged correct','Auto-judged incorrect','Manual review','Derivation availability','Median latency (s)','p95 latency (s)'],[
        [s,m['completed'],f"{m['coverage_pct']:.1f}%" if m['coverage_pct'] is not None else 'N/A',
         m['correct_cases_auto_judged'],m['incorrect_cases_auto_judged'],m['manual_review_cases'],
         f"{m['derivation_availability_pct']:.1f}%" if m['derivation_availability_pct'] is not None else 'N/A',
         f"{m['latency_median_s']:.6f}" if m['latency_median_s'] is not None else 'N/A',
         f"{m['latency_p95_s']:.6f}" if m['latency_p95_s'] is not None else 'N/A'] for s,m in summary.items()]),
       '', '## Weighted score and hard gates',
       'Weights: correctness 30%, derivation quality 30%, coverage 20%, reliability 10%, integration/maintainability 5%, performance 5%.',
       'Numeric weighted scores are withheld unless correctness and derivation components are defensibly scored. This run has no reviewed SmartSolver/StepSolver adapters and no automated derivation scorer, so **no eligible weighted score is claimed**.',
       'Hard-gate status: **NOT DETERMINED** until candidate source review, licence review, correctness checks, derivation audit, and execution-safety review are complete.',
       '', '## Category results']
for solver,cats in category_summary.items():
    lines += ['',f'### {solver}',md_table(['Category','Cases','Completed','Unavailable','Timeouts'],[[cat,v['cases'],v['completed'],v['unsupported_or_unavailable'],v['timeouts']] for cat,v in cats.items()])]
lines += ['', '## Reproducibility', f"- Output/status differences excluding timing: **{len(diffs)}**", '- See `reproducibility.json` for the exact comparison.',
          '', '## Interpretation and recommendation',
          '1. This initial execution validates the harness workflow and standalone SymPy isolation, but it is **not yet a fair three-way performance/quality comparison** because candidate-specific public API adapters were not configured.',
          '2. SymPy does not expose student-facing derivation traces through this adapter; its derivation availability is therefore low by design, not evidence that its symbolic results are poor.',
          '3. Review `repository_inspection.json` and `candidate_api_discovery.json`, inspect each project’s real step-generation code and licence, then implement reviewed adapters before drawing solver rankings.',
          '4. Add operation-specific mathematical comparators before interpreting correctness percentages. The harness deliberately marks complex solution sets, domain restrictions, ODEs, derivatives, integrals, inequalities, and structured outputs for manual review rather than falsely certifying them.',
          '5. Run the notebook twice on the same runtime for reproducibility; record hardware and commit IDs. Do not compare absolute latency across different Colab hardware sessions.',
          '', '## Artefacts', '- `cases.json`: dataset and metadata', '- `raw_results_run1.json` / `.csv`: row-level outputs and failures',
          '- `raw_results_run2.json`: second run', '- `summary_metrics.json` and `category_summary.json`: metrics',
          '- `failure_analysis.md`: failure taxonomy and reproduction details', '- `environment.json`, `installation_log.json`, `repository_inspection.json`, `candidate_api_discovery.json`',
          '', '## Reproduction command', 'Run all notebook cells from top to bottom in a fresh Google Colab runtime. Keep `INSTALL_CANDIDATES=False` until source review is complete. Set `CATEGORY` or `TEST_ID` in section 9 for a subset.']
(OUT/'summary.md').write_text('\n'.join(lines))
# Failure analysis
fail=['# Failure Analysis','',f'Generated UTC: {datetime.now(timezone.utc).isoformat()}','',
      'This report records observable failures and evaluation uncertainty. It does not reinterpret missing outputs as passes.','',
      '## Execution failures and unavailable systems']
for r in all_rows:
    if r['status']!='completed' or r.get('answer_correctness') in ('incorrect','manual_review'):
        fail += [f"### {r['solver']} — {r['test_id']}",f"- Status: {r['status']}",f"- Input: `{r['input']}`",
                 f"- Expected: `{r['expected']}`",f"- Actual: `{str(r.get('actual_answer'))[:1000]}`",
                 f"- Error/evaluator note: {r.get('error') or r.get('correctness_justification')}",
                 f"- Raw output: `{json.dumps(r.get('raw_output'),ensure_ascii=False,default=str)[:1600]}`",'']
fail += ['', '## Taxonomy','- `unsupported_or_unavailable`: no reviewed candidate adapter; this is not a solver mathematical failure.',
         '- `error` / `runtime_error`: execution or candidate exception; inspect the raw traceback and environment log.',
         '- `timeout`: process exceeded configured timeout.',
         '- `manual_review`: output was produced but automatic comparison did not certify mathematical equivalence.',
         '- `incorrect`: only used when a robust operation-specific comparator has explicitly proved the answer wrong; this starter notebook does not guess.',
         '', '## Current evaluator limitations','- Many operation types require operation-specific solution-set, derivative, integral, domain, or matrix comparators.',
         '- Domain restrictions for cancelled factors must be checked against the original expression.',
         '- The notebook does not automatically assign derivation marks because that requires inspecting actual candidate workings; it never invents steps.',
         '- Candidate source inspection and installation must be reviewed before treating a candidate as safely executable.']
(OUT/'failure_analysis.md').write_text('\n'.join(fail))
print('Wrote reports to',OUT)
print((OUT/'summary.md').read_text()[:8000])


## 12. Download artefacts

The output directory is `/content/math_solver_benchmark/output`. Zip the reports and dataset to download them from Colab. **Do not treat a successful notebook run as a completed solver comparison unless the candidate adapters, correctness evaluators, licence checks, and derivation reviews have been completed.**

In [ ]:
# Create a downloadable ZIP of all benchmark outputs, including raw row-level results.
import zipfile
zip_path = ROOT / 'math_solver_benchmark_artifacts.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in list(OUT.glob('*')) + list(DATA_DIR.glob('*')):
        if p.is_file(): z.write(p, arcname=str(p.relative_to(ROOT)))
print('ZIP:',zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print('Not running in Colab; retrieve the ZIP from:',zip_path)
